In [23]:
# Comments in English only
import math
import plotly.graph_objects as go


class CapacitorSymbolExact:
    def __init__(self, plate_gap=0.25, plate_half_width=0.45, line_color="black", w_std=2, w_plate=4):
        self.plate_gap = plate_gap
        self.plate_half_width = plate_half_width
        self.line_color = line_color
        self.w_std = w_std
        self.w_plate = w_plate

    def shapes_between(self, x, y_top_node, y_bot_node):
        if y_top_node <= y_bot_node:
            raise ValueError("y_top_node must be greater than y_bot_node")

        y_mid = 0.5 * (y_top_node + y_bot_node)
        y_top_plate = y_mid + 0.5 * self.plate_gap
        y_bot_plate = y_mid - 0.5 * self.plate_gap

        shapes = []
        shapes.append(self._line(x, y_top_node, x, y_top_plate, self.w_std))
        shapes.append(self._line(x, y_bot_plate, x, y_bot_node, self.w_std))
        shapes.append(self._line(x - self.plate_half_width, y_top_plate, x + self.plate_half_width, y_top_plate, self.w_plate))
        shapes.append(self._line(x - self.plate_half_width, y_bot_plate, x + self.plate_half_width, y_bot_plate, self.w_plate))
        return shapes

    def _line(self, x1, y1, x2, y2, width):
        return dict(type="line", x0=x1, y0=y1, x1=x2, y1=y2, line=dict(width=width, color=self.line_color))


class LadderBlock:
    def __init__(self, S=7, n_parallel=2, x0=0.0, y0=0.0, dx_string=2.2, dy_level=2.8, line_color="black", w_std=2, w_plate=4):
        self.S = S
        self.n_parallel = n_parallel
        self.x0 = x0
        self.y0 = y0
        self.dx_string = dx_string
        self.dy_level = dy_level

        self.line_color = line_color
        self.w_std = w_std

        self.cap = CapacitorSymbolExact(plate_gap=0.25, plate_half_width=0.45, line_color=line_color, w_std=w_std, w_plate=w_plate)

    def build(self):
        shapes = []

        xs = []
        k = 0
        while k < self.n_parallel:
            xs.append(self.x0 + k * self.dx_string)
            k = k + 1

        y_nodes = []
        i = 0
        while i < (self.S + 1):
            y_nodes.append(self.y0 - i * self.dy_level)
            i = i + 1

        j = 0
        while j < self.n_parallel:
            xj = xs[j]
            i = 0
            while i < self.S:
                shapes += self.cap.shapes_between(x=xj, y_top_node=y_nodes[i], y_bot_node=y_nodes[i + 1])
                i = i + 1
            j = j + 1

        x_left = xs[0]
        x_right = xs[-1]

        i = 0
        while i < (self.S + 1):
            yi = y_nodes[i]
            shapes.append(self._line(x_left, yi, x_right, yi))
            i = i + 1

        return {
            "shapes": shapes,
            "x_left": x_left,
            "x_right": x_right,
            "y_top": y_nodes[0],
            "y_bottom": y_nodes[-1],
            "y_nodes": y_nodes,
        }

    def _line(self, x1, y1, x2, y2):
        return dict(type="line", x0=x1, y0=y1, x1=x2, y1=y2, line=dict(width=self.w_std, color=self.line_color))


class HLeftLeg:
    def __init__(self, S=7, P=2, Pa=5, dx_string=2.2, dy_level=2.8, x_gap=8.0, line_color="black", w_std=2, w_plate=4):
        self.S = S
        self.P = P
        self.Pa = Pa
        self.x_gap = x_gap
        self.line_color = line_color
        self.w_std = w_std

        self.block_left = LadderBlock(S=S, n_parallel=P, dx_string=dx_string, dy_level=dy_level, line_color=line_color, w_std=w_std, w_plate=w_plate)
        self.block_right = LadderBlock(S=S, n_parallel=(Pa - P), dx_string=dx_string, dy_level=dy_level, line_color=line_color, w_std=w_std, w_plate=w_plate)

    def build(self, x0=0.0, y0=0.0):
        shapes = []

        self.block_left.x0 = x0
        self.block_left.y0 = y0
        b1 = self.block_left.build()
        shapes += b1["shapes"]

        self.block_right.x0 = x0 + self.x_gap
        self.block_right.y0 = y0
        b2 = self.block_right.build()
        shapes += b2["shapes"]

        y_top = b1["y_top"]
        y_bottom = b1["y_bottom"]

        x_left_bus = b1["x_right"]
        x_right_bus = b2["x_left"]

        shapes.append(self._line(x_left_bus, y_top, x_right_bus, y_top))
        shapes.append(self._line(x_left_bus, y_bottom, x_right_bus, y_bottom))

        return {
            "shapes": shapes,
            "x_left": min(b1["x_left"], b2["x_left"]),
            "x_right": max(b1["x_right"], b2["x_right"]),
            "y_top": y_top,
            "y_bottom": y_bottom,
            "y_nodes": b1["y_nodes"],
        }

    def _line(self, x1, y1, x2, y2):
        return dict(type="line", x0=x1, y0=y1, x1=x2, y1=y2, line=dict(width=self.w_std, color=self.line_color))


class HRightLeg:
    def __init__(self, Pt=9, P=2, S=7, dx_string=2.2, dy_level=2.8, x_gap_big=9.0, line_color="black", w_std=2, w_plate=4):
        self.Pt = Pt
        self.P = P
        self.S = S
        self.x_gap_big = x_gap_big
        self.line_color = line_color
        self.w_std = w_std

        n_strings_total = math.floor(Pt / P)
        self.n_big_groups = math.floor(n_strings_total / 2)

        self.blocks = []
        idx = 0
        while idx < self.n_big_groups:
            self.blocks.append(LadderBlock(S=S, n_parallel=P, dx_string=dx_string, dy_level=dy_level, line_color=line_color, w_std=w_std, w_plate=w_plate))
            idx = idx + 1

    def build(self, x0=0.0, y0=0.0):
        if self.n_big_groups < 1:
            raise ValueError("n_big_groups must be >= 1")

        shapes = []
        built = []

        idx = 0
        while idx < len(self.blocks):
            blk = self.blocks[idx]
            blk.x0 = x0 + idx * self.x_gap_big
            blk.y0 = y0
            b = blk.build()
            built.append(b)
            shapes += b["shapes"]
            idx = idx + 1

        y_top = built[0]["y_top"]
        y_bottom = built[0]["y_bottom"]

        x_min = built[0]["x_left"]
        x_max = built[0]["x_right"]

        idx = 1
        while idx < len(built):
            if built[idx]["x_left"] < x_min:
                x_min = built[idx]["x_left"]
            if built[idx]["x_right"] > x_max:
                x_max = built[idx]["x_right"]
            idx = idx + 1

        shapes.append(self._line(x_min, y_top, x_max, y_top))
        shapes.append(self._line(x_min, y_bottom, x_max, y_bottom))

        return {
            "shapes": shapes,
            "x_left": x_min,
            "x_right": x_max,
            "y_top": y_top,
            "y_bottom": y_bottom,
            "y_nodes": built[0]["y_nodes"],
        }

    def _line(self, x1, y1, x2, y2):
        return dict(type="line", x0=x1, y0=y1, x1=x2, y1=y2, line=dict(width=self.w_std, color=self.line_color))


class HCompleteWithNeutralCT:
    def __init__(
        self,
        Pt=9,
        Pa=5,
        P=2,
        S=7,
        St=3,
        dx_string=2.2,
        dy_level=2.8,
        gap_between_legs=14.0,
        left_internal_gap=8.0,
        right_group_gap=9.0,
        bus_stub=7.0,
        ct_box_w=3.2,
        ct_box_h=1.4,
        line_color="black",
        w_std=2,
        w_plate=4,
    ):
        self.Pt = Pt
        self.Pa = Pa
        self.P = P
        self.S = S
        self.St = St

        self.gap_between_legs = gap_between_legs
        self.bus_stub = bus_stub
        self.ct_box_w = ct_box_w
        self.ct_box_h = ct_box_h

        self.line_color = line_color
        self.w_std = w_std

        if (St < 1) or (St > (S - 1)):
            raise ValueError("St must satisfy 1 <= St <= S-1")

        self.left_leg = HLeftLeg(S=S, P=P, Pa=Pa, dx_string=dx_string, dy_level=dy_level, x_gap=left_internal_gap, line_color=line_color, w_std=w_std, w_plate=w_plate)
        self.right_leg = HRightLeg(Pt=Pt, P=P, S=S, dx_string=dx_string, dy_level=dy_level, x_gap_big=right_group_gap, line_color=line_color, w_std=w_std, w_plate=w_plate)

    def make_figure(self, x0=0.0, y0=0.0, width=1500, height=800, title=None):
        shapes = []
        annotations = []

        left = self.left_leg.build(x0=x0, y0=y0)
        shapes += left["shapes"]

        x_right_leg = left["x_right"] + self.gap_between_legs
        right = self.right_leg.build(x0=x_right_leg, y0=y0)
        shapes += right["shapes"]

        y_top = left["y_top"]
        y_bottom = left["y_bottom"]
        x_global_left = left["x_left"]
        x_global_right = right["x_right"]

        # Global top/bottom buses
        shapes.append(self._line(x_global_left, y_top, x_global_right, y_top))
        shapes.append(self._line(x_global_left, y_bottom, x_global_right, y_bottom))

        # Center stubs (top and bottom)
        x_center = 0.5 * (x_global_left + x_global_right)
        shapes.append(self._line(x_center, y_top, x_center, y_top + self.bus_stub))
        shapes.append(self._line(x_center, y_bottom, x_center, y_bottom - self.bus_stub))

        # Neutral line y from St: k = S - St
        k = self.S - self.St
        y_neutral = left["y_nodes"][k]

        # CT box coords
        x_ct0 = x_center - 0.5 * self.ct_box_w
        x_ct1 = x_center + 0.5 * self.ct_box_w
        y_ct0 = y_neutral - 0.5 * self.ct_box_h
        y_ct1 = y_neutral + 0.5 * self.ct_box_h

        # Neutral line split: stop at box edges (do not cross the box)
        shapes.append(self._line(x_global_left, y_neutral, x_ct0, y_neutral))
        shapes.append(self._line(x_ct1, y_neutral, x_global_right, y_neutral))

        # CT box
        shapes.append(
            dict(
                type="rect",
                x0=x_ct0,
                y0=y_ct0,
                x1=x_ct1,
                y1=y_ct1,
                line=dict(width=self.w_std, color=self.line_color),
                fillcolor="rgba(0,0,0,0)",
            )
        )
        annotations.append(
            dict(
                x=x_center,
                y=y_neutral,
                text="CT",
                showarrow=False,
                font=dict(size=18, color="black"),
                xanchor="center",
                yanchor="middle",
            )
        )

        # Limits
        pad_x = 6.0
        pad_y = 6.0
        x_min = x_global_left - pad_x
        x_max = x_global_right + pad_x
        y_min = y_bottom - self.bus_stub - pad_y
        y_max = y_top + self.bus_stub + pad_y

        fig = go.Figure()
        fig.update_layout(
            shapes=shapes,
            annotations=annotations,
            xaxis=dict(visible=False, range=[x_min, x_max]),
            yaxis=dict(visible=False, range=[y_min, y_max], scaleanchor="x", scaleratio=1.0),
            width=width,
            height=height,
            margin=dict(l=10, r=10, t=40 if title else 10, b=10),
            title=dict(text=title, x=0.5) if title else None,
        )
        return fig

    def _line(self, x1, y1, x2, y2):
        return dict(type="line", x0=x1, y0=y1, x1=x2, y1=y2, line=dict(width=self.w_std, color=self.line_color))


# Comments in English only
from dataclasses import dataclass
from typing import List, Dict


@dataclass
class ValidationResult:
    is_valid: bool
    errors: List[str]
    warnings: List[str]
    derived: Dict[str, int]


def validate_h_bridge_inputs(S: int, St: int, Pt: int, Pa: int, P: int) -> ValidationResult:
    """
    Validate inputs for the H-bridge capacitor bank drawing logic.

    Meanings:
      S  = total series groups
      St = series groups from H leg to neutral line (number of capacitors below neutral line)
      Pt = parallel units per phase (total)
      Pa = parallel units on left side of H
      P  = parallel units in affected string (base block width)

    Returns:
      ValidationResult with:
        - errors: must-fix issues
        - warnings: allowed but may change/degenerate the drawing
        - derived: useful computed counters used by the drawing
    """
    errors: List[str] = []
    warnings: List[str] = []
    derived: Dict[str, int] = {}

    # ---------------- Basic type/value checks ----------------
    for name, val in [("S", S), ("St", St), ("Pt", Pt), ("Pa", Pa), ("P", P)]:
        if not isinstance(val, int):
            errors.append(f"{name} must be an integer.")
        else:
            if val < 1:
                errors.append(f"{name} must be >= 1.")

    if errors:
        return ValidationResult(is_valid=False, errors=errors, warnings=warnings, derived=derived)

    # ---------------- Mandatory topology constraints ----------------
    if S < 2:
        errors.append("S must be >= 2 (otherwise St cannot satisfy 1 <= St <= S-1).")

    if St < 1 or St > (S - 1):
        errors.append("St must satisfy 1 <= St <= S-1.")

    if Pa >= Pt:
        errors.append("Pa must be < Pt (left side must be strictly smaller than total Pt).")

    if Pa < P:
        errors.append("Pa must be >= P (otherwise the left affected block cannot exist).")

    if Pt < (2 * P):
        errors.append("Pt must be >= 2*P (otherwise h_right_leg has zero big groups).")

    if Pa < 1 or (Pt - Pa) < 1:
        errors.append("Both sides must be >= 1: require 1 <= Pa <= Pt-1.")

    # If mandatory checks fail, stop here
    if errors:
        return ValidationResult(is_valid=False, errors=errors, warnings=warnings, derived=derived)

    # ---------------- Derived counters used by the drawing ----------------
    # Neutral line node index (0..S), node 0 is top, node S is bottom
    # k = S - St means St capacitors below the neutral line
    neutral_node_index = S - St

    # Left leg blocks: P and (Pa - P)
    left_block_P = P
    left_block_rest = Pa - P

    # Right-side affected-string count used earlier
    # nr_strings_right = floor((Pt - Pa) / P)
    nr_strings_right = (Pt - Pa) // P

    # Right leg big groups
    # n_big_groups = floor(floor(Pt/P)/2) = floor((Pt//P)/2)
    n_strings_total = Pt // P
    n_big_groups = n_strings_total // 2

    derived["neutral_node_index"] = neutral_node_index
    derived["left_block_P"] = left_block_P
    derived["left_block_rest"] = left_block_rest
    derived["nr_strings_right"] = nr_strings_right
    derived["n_strings_total"] = n_strings_total
    derived["n_big_groups"] = n_big_groups

    # ---------------- Warnings / soft constraints ----------------
    if left_block_rest < 1:
        warnings.append("Pa == P: (Pa - P) == 0, so the left leg becomes a single ladder block (no split).")

    if (Pt - Pa) < P:
        warnings.append("Pt - Pa < P: nr_strings_right becomes 0 (no right-side affected-string columns by that rule).")

    if (Pt % P) != 0:
        warnings.append("Pt is not a multiple of P: Pt//P discards a remainder (some parallel units are not represented).")

    if (n_strings_total % 2) != 0:
        warnings.append("Pt//P is odd: splitting into two equal big groups is not possible without asymmetry.")

    # Optional: you may want at least 1 big group
    if n_big_groups < 1:
        # This should be impossible here because Pt >= 2P is enforced, but keep as guard.
        warnings.append("n_big_groups computed as 0 (check Pt and P).")

    return ValidationResult(is_valid=True, errors=errors, warnings=warnings, derived=derived)


def format_validation_result(result: ValidationResult) -> str:
    """
    Human-readable multi-line output for Streamlit or console.
    """
    lines: List[str] = []
    lines.append(f"is_valid: {result.is_valid}")

    if len(result.errors) > 0:
        lines.append("errors:")
        for msg in result.errors:
            lines.append(f" - {msg}")
    else:
        lines.append("errors: (none)")

    if len(result.warnings) > 0:
        lines.append("warnings:")
        for msg in result.warnings:
            lines.append(f" - {msg}")
    else:
        lines.append("warnings: (none)")

    if len(result.derived) > 0:
        lines.append("derived:")
        for k in sorted(result.derived.keys()):
            lines.append(f" - {k}: {result.derived[k]}")
    else:
        lines.append("derived: (none)")

    return "\n".join(lines)


# ============================================================
# Validation helpers
# ============================================================
from dataclasses import dataclass
from typing import List, Dict
import math


@dataclass
class ValidationResult:
    is_valid: bool
    errors: List[str]
    warnings: List[str]
    derived: Dict[str, int]


def validate_h_bridge_inputs(S: int, St: int, Pt: int, Pa: int, P: int) -> ValidationResult:
    errors: List[str] = []
    warnings: List[str] = []
    derived: Dict[str, int] = {}

    # Basic type/value checks
    for name, val in [("S", S), ("St", St), ("Pt", Pt), ("Pa", Pa), ("P", P)]:
        if not isinstance(val, int):
            errors.append(f"{name} must be an integer.")
        else:
            if val < 1:
                errors.append(f"{name} must be >= 1.")

    if errors:
        return ValidationResult(is_valid=False, errors=errors, warnings=warnings, derived=derived)

    # Mandatory constraints
    if S < 2:
        errors.append("S must be >= 2 (otherwise St cannot satisfy 1 <= St <= S-1).")

    if St < 1 or St > (S - 1):
        errors.append("St must satisfy 1 <= St <= S-1.")

    if Pa >= Pt:
        errors.append("Pa must be < Pt (left side must be strictly smaller than total Pt).")

    if Pa < P:
        errors.append("Pa must be >= P (otherwise the left affected block cannot exist).")

    if Pt < (2 * P):
        errors.append("Pt must be >= 2*P (otherwise h_right_leg has zero big groups).")

    if Pa < 1 or (Pt - Pa) < 1:
        errors.append("Both sides must be >= 1: require 1 <= Pa <= Pt-1.")

    if errors:
        return ValidationResult(is_valid=False, errors=errors, warnings=warnings, derived=derived)

    # Derived values (useful for debugging)
    neutral_node_index = S - St
    n_strings_total = Pt // P
    n_big_groups = n_strings_total // 2
    nr_strings_right = (Pt - Pa) // P
    left_block_rest = Pa - P

    derived["neutral_node_index"] = neutral_node_index
    derived["n_strings_total"] = n_strings_total
    derived["n_big_groups"] = n_big_groups
    derived["nr_strings_right"] = nr_strings_right
    derived["left_block_rest"] = left_block_rest

    # Soft constraints / warnings
    if left_block_rest < 1:
        warnings.append("Pa == P: (Pa - P) == 0, so the left leg becomes a single ladder block (no split).")

    if (Pt - Pa) < P:
        warnings.append("Pt - Pa < P: floor((Pt - Pa)/P) becomes 0 (no right-side affected-string columns by that rule).")

    if (Pt % P) != 0:
        warnings.append("Pt is not a multiple of P: Pt//P discards a remainder (some parallel units are not represented).")

    if (n_strings_total % 2) != 0:
        warnings.append("Pt//P is odd: splitting into two equal big groups is not possible without asymmetry.")

    return ValidationResult(is_valid=True, errors=errors, warnings=warnings, derived=derived)


def print_validation_messages(result: ValidationResult) -> None:
    if result.is_valid:
        if len(result.warnings) > 0:
            print("Warnings:")
            for msg in result.warnings:
                print(" - " + msg)
        return

    print("Invalid input parameters:")
    for msg in result.errors:
        print(" - " + msg)

    if len(result.warnings) > 0:
        print("Warnings:")
        for msg in result.warnings:
            print(" - " + msg)


# ============================================================
# Your main program (runs only if validation passes)
# ============================================================

Pt = 9
Pa = 5
P = 2
S = 7
St = 3

result = validate_h_bridge_inputs(S=S, St=St, Pt=Pt, Pa=Pa, P=P)
print_validation_messages(result)

if result.is_valid:
    h = HCompleteWithNeutralCT(
        Pt=Pt,
        Pa=Pa,
        P=P,
        S=S,
        St=St,
        gap_between_legs=14.0,
        left_internal_gap=8.0,
        right_group_gap=9.0,
        bus_stub=7.0,
        ct_box_w=3.2,
        ct_box_h=1.4,
    )

    fig = h.make_figure(title=f"Neutral line touches CT box only (St={St})")
    fig.show()



Warnings:
 - Pt is not a multiple of P: Pt//P discards a remainder (some parallel units are not represented).
